# 04 - Reorganization Analysis
**Core thesis contribution: Error-as-Signal Analysis**

This notebook implements your novel 'error-as-signal' framework:
- Train classifiers on resting-state fMRI
- Apply to task-based fMRI
- Misclassifications indicate functional reorganization

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (15, 10)

In [ ]:
# Setup paths
PROJECT_ROOT = Path.cwd().parent if 'analysis' in str(Path.cwd()) else Path.cwd()
RESULTS_DIR = PROJECT_ROOT / 'data' / 'results'
ANALYSIS_DIR = PROJECT_ROOT / 'data' / 'analysis'
FIGURES_DIR = PROJECT_ROOT / 'figures' / 'reorganization'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Results: {RESULTS_DIR}")
print(f"Analysis: {ANALYSIS_DIR}")
print(f"Figures: {FIGURES_DIR}")

## 1. Load Results for Reorganization Analysis

In [ ]:
# Load region-level results
region_perf = pd.read_csv(ANALYSIS_DIR / 'compiled_region_performance.csv')
region_cv = region_perf[region_perf['Phase'] == 'CV'].copy()
region_task = region_perf[region_perf['Phase'] == 'Task'].copy()

print(f"CV regions: {len(region_cv)}")
print(f"Task regions: {len(region_task)}")

In [ ]:
# Load predictions for detailed analysis
cv_predictions = np.load(RESULTS_DIR / 'full_connectivity_analysis' / 'multinomial' / 'cv_predictions.npy')
cv_true_labels = np.load(RESULTS_DIR / 'full_connectivity_analysis' / 'multinomial' / 'cv_true_labels.npy')
task_predictions = np.load(RESULTS_DIR / 'full_connectivity_analysis' / 'task_testing' / 'task_predictions.npy')
task_true_labels = np.load(RESULTS_DIR / 'full_connectivity_analysis' / 'task_testing' / 'task_true_labels.npy')

print(f"\nCV predictions shape: {cv_predictions.shape}")
print(f"Task predictions shape: {task_predictions.shape}")

## 2. Calculate Reorganization Metrics

In [ ]:
# Merge CV and Task performance per region
reorganization_df = pd.merge(
    region_cv[['Region', 'Network', 'Accuracy', 'F1_Score']],
    region_task[['Region', 'Accuracy', 'F1_Score']],
    on='Region',
    suffixes=('_CV', '_Task')
)

# Calculate reorganization score (performance drop)
reorganization_df['Accuracy_Drop'] = reorganization_df['Accuracy_CV'] - reorganization_df['Accuracy_Task']
reorganization_df['F1_Drop'] = reorganization_df['F1_Score_CV'] - reorganization_df['F1_Score_Task']

# Positive drop = reorganization (worse performance on task)
# Negative drop = stable or improved (no reorganization)
reorganization_df['Reorganization_Score'] = reorganization_df['Accuracy_Drop']

print("\nReorganization Statistics:")
print(f"  Mean reorganization score: {reorganization_df['Reorganization_Score'].mean():.4f}")
print(f"  Std: {reorganization_df['Reorganization_Score'].std():.4f}")
print(f"  Regions with positive reorganization: {(reorganization_df['Reorganization_Score'] > 0).sum()}")
print(f"  Regions with negative reorganization: {(reorganization_df['Reorganization_Score'] < 0).sum()}")

In [ ]:
# Identify most reorganized regions
print("\nTop 20 Most Reorganized Regions (largest performance drop):")
most_reorganized = reorganization_df.nlargest(20, 'Reorganization_Score')
print(most_reorganized[['Region', 'Network', 'Accuracy_CV', 'Accuracy_Task', 'Reorganization_Score']].to_string(index=False))

In [ ]:
# Identify least reorganized regions (stable performance)
print("\nTop 20 Least Reorganized Regions (most stable):")
least_reorganized = reorganization_df.nsmallest(20, 'Reorganization_Score')
print(least_reorganized[['Region', 'Network', 'Accuracy_CV', 'Accuracy_Task', 'Reorganization_Score']].to_string(index=False))

## 3. Network-Level Reorganization

In [ ]:
# Aggregate reorganization by network
network_reorganization = reorganization_df.groupby('Network').agg({
    'Reorganization_Score': ['mean', 'std', 'median', 'min', 'max'],
    'Accuracy_Drop': ['mean', 'std'],
    'Region': 'count'
}).round(4)

network_reorganization.columns = ['_'.join(col).strip() for col in network_reorganization.columns.values]
network_reorganization = network_reorganization.sort_values('Reorganization_Score_mean', ascending=False)

print("\nNetwork-Level Reorganization:")
print(network_reorganization)

In [ ]:
# Plot network reorganization
fig, ax = plt.subplots(figsize=(12, 6))

networks = network_reorganization.index
means = network_reorganization['Reorganization_Score_mean']
stds = network_reorganization['Reorganization_Score_std']

colors = ['red' if x > 0 else 'green' for x in means]
bars = ax.bar(networks, means, yerr=stds, capsize=5, color=colors, 
              alpha=0.7, edgecolor='black', linewidth=1.5)

ax.axhline(y=0, color='black', linestyle='-', linewidth=2)
ax.set_xlabel('Network', fontsize=12)
ax.set_ylabel('Mean Reorganization Score', fontsize=12)
ax.set_title('Functional Reorganization by Network\n(Positive = Reorganization, Negative = Stable)', 
            fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'network_reorganization.png', dpi=300, bbox_inches='tight')
print("\nSaved network reorganization plot")
plt.show()

## 4. Visualization: Reorganization Distribution

In [ ]:
# Plot reorganization distribution
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Overall distribution
ax = axes[0, 0]
ax.hist(reorganization_df['Reorganization_Score'], bins=50, edgecolor='black', alpha=0.7)
ax.axvline(0, color='red', linestyle='--', linewidth=2, label='No reorganization')
ax.axvline(reorganization_df['Reorganization_Score'].mean(), color='blue', 
          linestyle='--', linewidth=2, label=f'Mean: {reorganization_df["Reorganization_Score"].mean():.3f}')
ax.set_xlabel('Reorganization Score', fontsize=11)
ax.set_ylabel('Number of Regions', fontsize=11)
ax.set_title('Distribution of Reorganization Scores', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)

# By network
ax = axes[0, 1]
reorganization_df.boxplot(column='Reorganization_Score', by='Network', ax=ax)
ax.axhline(0, color='red', linestyle='--', linewidth=1.5, alpha=0.7)
ax.set_xlabel('Network', fontsize=11)
ax.set_ylabel('Reorganization Score', fontsize=11)
ax.set_title('Reorganization by Network', fontsize=12, fontweight='bold')
plt.sca(ax)
plt.xticks(rotation=45, ha='right')
plt.suptitle('')  # Remove auto title

# Scatter: CV vs Task accuracy
ax = axes[1, 0]
scatter = ax.scatter(reorganization_df['Accuracy_CV'], 
                    reorganization_df['Accuracy_Task'],
                    c=reorganization_df['Reorganization_Score'],
                    cmap='RdYlGn_r', alpha=0.6, s=50, edgecolors='black', linewidth=0.5)
ax.plot([0, 1], [0, 1], 'k--', linewidth=2, alpha=0.5, label='Perfect generalization')
ax.set_xlabel('CV Accuracy', fontsize=11)
ax.set_ylabel('Task Accuracy', fontsize=11)
ax.set_title('CV vs Task Performance', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Reorganization Score', fontsize=10)

# Violin plot by network
ax = axes[1, 1]
import seaborn as sns
sns.violinplot(data=reorganization_df, x='Network', y='Reorganization_Score', ax=ax)
ax.axhline(0, color='red', linestyle='--', linewidth=1.5, alpha=0.7)
ax.set_xlabel('Network', fontsize=11)
ax.set_ylabel('Reorganization Score', fontsize=11)
ax.set_title('Reorganization Distribution by Network', fontsize=12, fontweight='bold')
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'reorganization_distributions.png', dpi=300, bbox_inches='tight')
print("\nSaved reorganization distributions plot")
plt.show()

## 5. Statistical Tests

In [ ]:
from scipy import stats

# Test if reorganization scores differ significantly from zero
t_stat, p_value = stats.ttest_1samp(reorganization_df['Reorganization_Score'], 0)
print("\nOne-sample t-test (H0: reorganization score = 0):")
print(f"  t-statistic: {t_stat:.4f}")
print(f"  p-value: {p_value:.4e}")
print(f"  Significant: {'Yes' if p_value < 0.05 else 'No'}")

# Test for differences between networks
from scipy.stats import f_oneway

network_groups = [group['Reorganization_Score'].values 
                 for name, group in reorganization_df.groupby('Network')]

f_stat, p_value_anova = f_oneway(*network_groups)
print("\nOne-way ANOVA (H0: no difference between networks):")
print(f"  F-statistic: {f_stat:.4f}")
print(f"  p-value: {p_value_anova:.4e}")
print(f"  Significant: {'Yes' if p_value_anova < 0.05 else 'No'}")

## 6. Save Results

In [ ]:
# Save reorganization results
reorganization_df.to_csv(ANALYSIS_DIR / 'reorganization_analysis.csv', index=False)
network_reorganization.to_csv(ANALYSIS_DIR / 'network_reorganization.csv')
most_reorganized.to_csv(ANALYSIS_DIR / 'most_reorganized_regions.csv', index=False)
least_reorganized.to_csv(ANALYSIS_DIR / 'least_reorganized_regions.csv', index=False)

# Save statistical results
stats_results = {
    'one_sample_ttest': {
        't_statistic': float(t_stat),
        'p_value': float(p_value),
        'significant': bool(p_value < 0.05)
    },
    'anova': {
        'f_statistic': float(f_stat),
        'p_value': float(p_value_anova),
        'significant': bool(p_value_anova < 0.05)
    },
    'summary': {
        'mean_reorganization': float(reorganization_df['Reorganization_Score'].mean()),
        'std_reorganization': float(reorganization_df['Reorganization_Score'].std()),
        'median_reorganization': float(reorganization_df['Reorganization_Score'].median()),
        'regions_with_reorganization': int((reorganization_df['Reorganization_Score'] > 0).sum()),
        'total_regions': len(reorganization_df)
    }
}

with open(ANALYSIS_DIR / 'reorganization_statistics.json', 'w') as f:
    json.dump(stats_results, f, indent=2)

print("\n" + "="*80)
print("REORGANIZATION ANALYSIS COMPLETE")
print("="*80)
print("\nThis is your core thesis contribution!")
print("\nSaved files:")
print("  - reorganization_analysis.csv")
print("  - network_reorganization.csv")
print("  - most_reorganized_regions.csv")
print("  - least_reorganized_regions.csv")
print("  - reorganization_statistics.json")
print("\nSaved figures:")
print("  - network_reorganization.png")
print("  - reorganization_distributions.png")